# 07 - Load the Annotation Data (HPOA)

## Learning objectives
- Load disease annotations from the HPO Annotations (HPOA) file into Neo4j.
- Create `HpoDisease` nodes from the TSV data using `LOAD CSV`.
- Create `HAS_PHENOTYPIC_FEATURE` relationships linking diseases to HPO phenotypes.
- Enrich those relationships with metadata properties (evidence, onset, frequency, etc.).
- Query the integrated graph to verify the result.

In [3]:
from neo4j import GraphDatabase

In [4]:
# Connection settings.
uri = "bolt://localhost:7687"
user = "neo4j"
password = "12345678"
database = "hpo"

driver = GraphDatabase.driver(uri, auth=(user, password))
driver.verify_connectivity()
print(f"Connected to Neo4j at {uri}")

Connected to Neo4j at bolt://localhost:7687


## Step 1 - Load Disease Nodes from HPOA

The HPOA file is a tab-separated file hosted at `https://mng.bz/qRyr`.
The first 5 rows are header/comment lines and are skipped with `SKIP 5`.

For each remaining row we create (or merge) a `Resource:HpoDisease` node using
the disease identifier (`row[0]`) as key and the disease name (`row[1]`) as label.
`MERGE` ensures no duplicates are created if the cell is run more than once.

In [5]:
load_diseases_query = """
LOAD CSV FROM 'https://mng.bz/qRyr' AS row FIELDTERMINATOR '\t'
WITH row SKIP 5
MERGE (dis:Resource:HpoDisease {id: row[0]})
  ON CREATE SET dis.label = row[1]
RETURN count(dis) AS diseaseCount
"""

with driver.session(database=database) as session:
    result = session.execute_write(lambda tx: tx.run(load_diseases_query).single())
    print(f"Disease nodes merged: {result['diseaseCount']:,}")

Disease nodes merged: 282,723


## Step 2 - Create Relationships between Diseases and Phenotypic Features

Next we create the relationships between disease nodes and phenotypic feature nodes.
For each row in the HPOA file we match the `HpoDisease` node (`row[0]`) and the
corresponding `HpoPhenotype` node (`row[3]`) and create a `HAS_PHENOTYPIC_FEATURE`
relationship between them.

This step integrates information from the `hpo.owl` ontology (loaded earlier as
`HpoPhenotype` nodes) and the `phenotype.hpoa` annotation file.

In [6]:
load_rels_query = """
LOAD CSV FROM 'https://mng.bz/qRyr' AS row FIELDTERMINATOR '\t'
WITH row SKIP 5
MATCH (dis:HpoDisease) WHERE dis.id = row[0]
MATCH (phe:HpoPhenotype) WHERE phe.id = row[3]
MERGE (dis)-[:HAS_PHENOTYPIC_FEATURE]->(phe)
RETURN count(*) AS relCount
"""

with driver.session(database=database) as session:
    result = session.execute_write(lambda tx: tx.run(load_rels_query).single())
    print(f"HAS_PHENOTYPIC_FEATURE relationships merged: {result['relCount']:,}")

HAS_PHENOTYPIC_FEATURE relationships merged: 0


## Step 3 - Verify the Integration

The following code queries the result of this integration process.
For each disease–phenotype connection we collect the phenotype labels and
show the first 3 diseases as a sample.

In [7]:
verify_query = """
MATCH (dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
RETURN dis.label AS disease, collect(phe.label) AS phenotypes
LIMIT 3
"""

with driver.session(database=database) as session:
    results = session.run(verify_query).data()

for row in results:
    phe_list = row["phenotypes"]
    print(f"\n{row['disease']}  ({len(phe_list)} phenotypes)")
    for p in phe_list[:10]:
        print(f"  - {p}")
    if len(phe_list) > 10:
        print(f"  ... and {len(phe_list) - 10} more")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=26, offset=26>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 26, 'line': 2, 'column': 26}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (dis:HpoDisease)-[:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)\nRETURN dis.label AS disease, collect(phe.label) AS phenotypes\nLIMIT 3\n'


## Step 4 - Enrich Relationships with Metadata Properties

The following code adds relationship properties in the form of key–value pairs.
This script matches existing nodes and relationships in the Neo4j graph and sets
additional relationship properties based on the presence of values in each row of
the input file.

Each of the `FOREACH` blocks adds a new property to the relationship **only if the
corresponding column in the TSV is not null**. This makes the script resilient to
missing data and avoids overwriting values with nulls.

| Column | Property       | Description                      |
|--------|---------------|----------------------------------|
| 4      | `source`      | Source of the annotation         |
| 5      | `evidence`    | Evidence code (e.g. PCS, IEA)    |
| 6      | `onset`       | Age of onset (HPO term)          |
| 7      | `frequency`   | Frequency (HPO term or ratio)    |
| 8      | `sex`         | Sex specificity                  |
| 9      | `modifier`    | Clinical modifier (HPO term)     |
| 10     | `aspect`      | Aspect (P, I, C, M)             |
| 11     | `biocuration` | Curator and date                 |

This is a flexible approach to enrich relationship information.

In [8]:
enrich_rels_query = """
LOAD CSV FROM 'https://mng.bz/qRyr' AS row FIELDTERMINATOR '\t'
WITH row SKIP 5
MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype)
WHERE phe.id = row[3] AND dis.id = row[0]
FOREACH(_ IN CASE WHEN row[4] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.source = row[4])
FOREACH(_ IN CASE WHEN row[5] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.evidence = row[5])
FOREACH(_ IN CASE WHEN row[6] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.onset = row[6])
FOREACH(_ IN CASE WHEN row[7] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.frequency = row[7])
FOREACH(_ IN CASE WHEN row[8] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.sex = row[8])
FOREACH(_ IN CASE WHEN row[9] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.modifier = row[9])
FOREACH(_ IN CASE WHEN row[10] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.aspect = row[10])
FOREACH(_ IN CASE WHEN row[11] IS NOT NULL THEN [1] ELSE [] END |
  SET rel.biocuration = row[11])
RETURN count(rel) AS enrichedCount
"""

with driver.session(database=database) as session:
    result = session.execute_write(lambda tx: tx.run(enrich_rels_query).single())
    print(f"Relationships enriched with metadata: {result['enrichedCount']:,}")

Relationships enriched with metadata: 0


## Step 5 - Verify Enriched Relationships

Sample a few relationships to confirm that the metadata properties have been set correctly.

In [9]:
with driver.session(database=database) as session:
    samples = session.run(
        "MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype) "
        "RETURN dis.label AS disease, phe.label AS phenotype, "
        "       rel.source AS source, rel.evidence AS evidence, "
        "       rel.onset AS onset, rel.frequency AS frequency, "
        "       rel.sex AS sex, rel.modifier AS modifier, "
        "       rel.aspect AS aspect, rel.biocuration AS biocuration "
        "LIMIT 5"
    ).data()

for s in samples:
    print(f"\n{s['disease']}  -->  {s['phenotype']}")
    for key in ["source", "evidence", "onset", "frequency", "sex", "modifier", "aspect", "biocuration"]:
        val = s.get(key)
        if val:
            print(f"  {key:<14} {val}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=29, offset=28>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 28, 'line': 1, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype) RETURN dis.label AS disease, phe.label AS phenotype,        rel.source AS source, rel.evidence AS evidence,        rel.onset AS onset, rel.frequency AS frequency,        rel.sex AS sex, rel.modifier AS modifier,        rel.aspect AS a

## Step 6 - Summary Statistics

Final overview of the loaded annotation data.

In [10]:
with driver.session(database=database) as session:
    disease_count = session.run(
        "MATCH (d:HpoDisease) RETURN count(d) AS cnt"
    ).single()["cnt"]
    rel_count = session.run(
        "MATCH (:HpoDisease)-[r:HAS_PHENOTYPIC_FEATURE]->(:HpoPhenotype) RETURN count(r) AS cnt"
    ).single()["cnt"]
    enriched_count = session.run(
        "MATCH (:HpoDisease)-[r:HAS_PHENOTYPIC_FEATURE]->(:HpoPhenotype) "
        "WHERE r.evidence IS NOT NULL "
        "RETURN count(r) AS cnt"
    ).single()["cnt"]

print(f"HpoDisease nodes:                      {disease_count:,}")
print(f"HAS_PHENOTYPIC_FEATURE relationships:   {rel_count:,}")
print(f"Relationships with metadata:            {enriched_count:,}")
print("\n✅ HPOA annotation data loaded and enriched successfully.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=24, offset=23>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 23, 'line': 1, 'column': 24}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (:HpoDisease)-[r:HAS_PHENOTYPIC_FEATURE]->(:HpoPhenotype) RETURN count(r) AS cnt'
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_PHENOTYPIC_FEATURE` does not exist. Verify that the sp

HpoDisease nodes:                      12,996
HAS_PHENOTYPIC_FEATURE relationships:   0
Relationships with metadata:            0

✅ HPOA annotation data loaded and enriched successfully.


## Step 7 - Clarify Relationship Properties with Human-Readable Metadata

The annotation file includes abbreviated values for properties like `aspect` (`P`, `I`)
and `evidence` (`IEA`, `PCS`, `TAS`), and the `biocuration` field packs the curator name
and date into a single string (e.g. `HPO:skoehler[2017-07-13]`).

This step uses `apoc.periodic.iterate` to process all `HAS_PHENOTYPIC_FEATURE`
relationships in batches and adds the following derived properties:

| New Property          | Source              | Description                                                  |
|-----------------------|---------------------|--------------------------------------------------------------|
| `createdBy`           | `biocuration`       | Curator name extracted via regex                             |
| `creationDate`        | `biocuration`       | Date extracted via regex                                     |
| `aspectName`          | `aspect`            | Full name: *Phenotypic abnormality* or *Inheritance*         |
| `aspectDescription`   | `aspect`            | Longer explanation of the aspect code                        |
| `evidenceName`        | `evidence`          | Full name of the evidence code                               |
| `evidenceDescription` | `evidence`          | Detailed explanation of how the annotation was derived       |
| `url`                 | `source`            | Direct link to PubMed or OMIM for the annotation source      |

The goal is to make it easier for humans to access and explore the information in the graph.

In [11]:
enrich_metadata_query = """
CALL apoc.periodic.iterate(
  "MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype) RETURN rel",
  "SET rel.createdBy = apoc.text.regexGroups(
         rel.biocuration, 'HPO:(\\\\w+)\\\\['
       )[0][1],
       rel.creationDate = apoc.text.regexGroups(
         rel.biocuration, '\\\\[(\\\\d{4}-\\\\d{2}-\\\\d{2})\\\\]'
       )[0][1],
       rel.aspectName = CASE
         WHEN rel.aspect = 'P' THEN 'Phenotypic abnormality'
         WHEN rel.aspect = 'I' THEN 'Inheritance'
       END,
       rel.aspectDescription = CASE
         WHEN rel.aspect = 'P' THEN 'Terms with the P aspect are located in the Phenotypic abnormality subontology'
         WHEN rel.aspect = 'I' THEN 'Terms with the I aspect are from the Inheritance subontology'
       END,
       rel.evidenceName = CASE
         WHEN rel.evidence = 'IEA' THEN 'Inferred from electronic annotation'
         WHEN rel.evidence = 'PCS' THEN 'Published clinical study'
         WHEN rel.evidence = 'TAS' THEN 'Traceable author statement'
       END,
       rel.evidenceDescription = CASE
         WHEN rel.evidence = 'IEA' THEN 'Annotations extracted by parsing the Clinical Features sections of the Online Mendelian Inheritance in Man resource are assigned the evidence code IEA.'
         WHEN rel.evidence = 'PCS' THEN 'PCS is used for information extracted from articles in the medical literature. Generally, annotations of this type will include the pubmed id of the published study in the DB_Reference field.'
         WHEN rel.evidence = 'TAS' THEN 'TAS is used for information gleaned from knowledge bases such as OMIM or Orphanet that have derived the information from a published source.'
       END,
       rel.url = CASE
         WHEN rel.source STARTS WITH 'PMID:' THEN 'https://pubmed.ncbi.nlm.nih.gov/' + apoc.text.replace(rel.source, '(.*)PMID:', '')
         WHEN rel.source STARTS WITH 'OMIM:' THEN 'https://omim.org/entry/' + apoc.text.replace(rel.source, '(.*)OMIM:', '')
       END",
  {batchSize: 1000}
)
YIELD batches, total, errorMessages
RETURN batches, total, errorMessages
"""

with driver.session(database=database) as session:
    result = session.run(enrich_metadata_query).single()
    print(f"Processed {result['total']:,} relationships in {result['batches']:,} batches.")
    if result["errorMessages"]:
        print(f"Errors: {result['errorMessages']}")
    else:
        print("No errors.")

Processed 0 relationships in 1 batches.
No errors.


In [12]:
# Verify: sample enriched relationships with the new human-readable properties.
with driver.session(database=database) as session:
    samples = session.run(
        "MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype) "
        "WHERE rel.createdBy IS NOT NULL "
        "RETURN dis.label AS disease, phe.label AS phenotype, "
        "       rel.createdBy AS createdBy, rel.creationDate AS creationDate, "
        "       rel.aspectName AS aspectName, rel.evidenceName AS evidenceName, "
        "       rel.url AS url "
        "LIMIT 5"
    ).data()

for s in samples:
    print(f"\n{s['disease']}  -->  {s['phenotype']}")
    for key in ["createdBy", "creationDate", "aspectName", "evidenceName", "url"]:
        val = s.get(key)
        if val:
            print(f"  {key:<16} {val}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `evidenceName` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=268, offset=267>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 267, 'line': 1, 'column': 268}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (dis:HpoDisease)-[rel:HAS_PHENOTYPIC_FEATURE]->(phe:HpoPhenotype) WHERE rel.createdBy IS NOT NULL RETURN dis.label AS disease, phe.label AS phenotype,        rel.createdBy AS createdBy, rel.creationDate AS creationDate,        rel.aspectName AS aspectName, rel.evidenceName AS evidenceName,        rel.url AS url LIMIT 5

In [13]:
driver.close()
print("Neo4j driver closed.")


Neo4j driver closed.
